<a href="https://colab.research.google.com/github/vk731843-code/Energy-prediction/blob/main/single_nureal_networke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

Load the data set

In [ ]:
df = pd.read_csv('/content/climate_energy.csv')
df.head()

,timestamp,location,temperature,energy_consumption
0,2024-01-01 00:00:00,Chennai,21.1,42.27
1,2024-01-01 01:00:00,Salem,20.8,41.64
2,2024-01-01 02:00:00,Madurai,21.7,41.88
3,2024-01-01 03:00:00,Coimbatore,19.6,43.51
4,2024-01-01 04:00:00,Coimbatore,20.9,42.29


Data set information

In [ ]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   timestamp           2000 non-null   object 
 1   location            2000 non-null   object 
 2   temperature         2000 non-null   float64
 3   energy_consumption  2000 non-null   float64
dtypes: float64(2), object(2)
memory usage: 62.6+ KB
None
       temperature  energy_consumption
count  2000.000000         2000.000000
mean     26.476500           66.403965
std       4.153513           21.389659
min      16.700000           20.560000
25%      23.200000           47.997500
50%      26.500000           65.225000
75%      29.700000           82.810000
max      36.600000          124.310000


convert timestamp into numerical features

In [ ]:
df["timestamp"] = pd.to_datetime(df ["timestamp"])
df["hour"]=df["timestamp"].dt.hour
df["day"]= df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month

df.drop("timestamp", axis=1, inplace=True)

print(df.head())

     location  temperature  energy_consumption  hour  day  month
0     Chennai         21.1               42.27     0    1      1
1       Salem         20.8               41.64     1    1      1
2     Madurai         21.7               41.88     2    1      1
3  Coimbatore         19.6               43.51     3    1      1
4  Coimbatore         20.9               42.29     4    1      1


In [ ]:
encoder = LabelEncoder()

df ["location"] = encoder.fit_transform(df ["location"])

print(df.head())

   location  temperature  energy_consumption  hour  day  month
0         0         21.1               42.27     0    1      1
1         3         20.8               41.64     1    1      1
2         2         21.7               41.88     2    1      1
3         1         19.6               43.51     3    1      1
4         1         20.9               42.29     4    1      1


Split features and target

In [ ]:
x = df.drop("energy_consumption", axis=1)
y = df["energy_consumption"]

Normalize Features

In [ ]:
scaler = StandardScaler()
x = scaler.fit_transform(x)

Train - Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)


(1600, 5)
(400, 5)


Buildd Feedforward Neural Network

In [ ]:
model = Sequential()

model.add(Dense(16, activation="relu", input_shape=(x_train.shape[1],))) #input layer

model.add(Dense(8, activation="relu"))

model.add(Dense(1))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compile Model

In [ ]:
model.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])

view model sumary

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

Train the model

In [ ]:
history = model.fit(x_train, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=1)

Epoch 1/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 4912.1475 - mae: 66.7905 - val_loss: 4681.5283 - val_mae: 64.8220
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4739.4453 - mae: 65.4701 - val_loss: 4416.1465 - val_mae: 62.7822
Epoch 3/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4297.3057 - mae: 62.0861 - val_loss: 3776.0769 - val_mae: 57.7101
Epoch 4/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 3430.6909 - mae: 54.9957 - val_loss: 2742.5347 - val_mae: 48.5834
Epoch 5/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2250.0183 - mae: 43.7665 - val_loss: 1544.2517 - val_mae: 35.6590
Epoch 6/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1111.2493 - mae: 29.8802 - val_loss: 629.2397 - val_mae: 22.1660
Epoch 7/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 413.4528 - mae: 17.5760 - val_loss: 235.6260 - val_mae: 12.9124
Epoch 8/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 173.1983 - mae: 10.7291 - val_loss: 139.8224 - val_mae: 9.8698
Epoch 

evaluate the model

In [ ]:
loss , mae= model.evaluate(x_test, y_test)

print("Test Loss :", loss)
print("Test MAE :", mae)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 23.8033 - mae: 3.9241  
Test Loss : 23.803274154663086
Test MAE : 3.9241061210632324


Make predictions

In [ ]:
predictions = model.predict(x_test)

print(predictions [:5])

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
[[110.55004 ]
 [ 59.63689 ]
 [ 84.79302 ]
 [ 72.034096]
 [ 80.91956 ]]


Calculate Performance matrics

In [ ]:
mse=mean_squared_error(y_test, predictions)
rmse=np.sqrt(mse)
mae=mean_absolute_error(y_test, predictions)
r2=r2_score(y_test, predictions)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:",rmse)
print("Mean Absolute Error:", mae)
print("R2 Score:",r2)

Mean Squared Error: 23.803272378023088
Root Mean Squared Error: 4.878859741581334
Mean Absolute Error: 3.9241060497283935
R2 Score: 0.9484375159581252
